In [ ]:
import cv2
import matplotlib.pyplot as plt
import glob
import numpy as np

Ensure inline plots

In [ ]:
%matplotlib inline  

Function to load and preprocess images

In [ ]:
def load_images(folder):
    images = []
    for file in glob.glob(folder + "/*.jpg"):
        img = cv2.imread(file)
        if img is not None:  # Ensure image is read correctly
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
            images.append(img)
    return images

Load Organic and Recyclable Images For Training

In [ ]:
O_img = load_images("DATASET/TRAIN/O")  # Organic images
R_img = load_images("DATASET/TRAIN/R")  # Recyclable images 

Load Organic and Recyclable Images For Testing

In [ ]:
O_test = load_images("DATASET/TEST/O")  # Organic images (Testing)
R_test = load_images("DATASET/TEST/R")  # Recyclable images (Testing

Combine training images and labels

In [ ]:
train_img = O_train + R_train
train_labels = np.array([0] * len(O_train) + [1] * len(R_train))  # 0 = Organic, 1 = Recyclable

Combine testing images and labels

In [ ]:
test_img = O_test + R_test
test_labels = np.array([0] * len(O_test) + [1] * len(R_test))  # 0 = Organic, 1 = Recyclable

Print summary

In [ ]:
print(f"Loaded {len(O_img)} organic images and {len(R_img)} recyclable images.")
print(f"Total training images: {len(train_img)}")
# Print dataset size
print(f"Training Samples: {len(train_img)} | Testing Samples: {len(test_img)}")

In [ ]:
import random

# Define the number of sample images to display
num_samples = 25

# Randomly select images and their corresponding labels
random_indices = random.choices(range(len(train_img)), k=num_samples)
sample_images = [train_img[i] for i in random_indices]
sample_labels = [train_labels[i] for i in random_indices]  # 0 = Organic, 1 = Recyclable

# Set up figure for plotting
fig = plt.figure(figsize=(20, 20))

for i, (img, label) in enumerate(zip(sample_images, sample_labels), 1):
    ax = fig.add_subplot(5, 5, i)
    ax.imshow(img)
    ax.axis("off")  # Hide axis
    title = "O" if label == 0 else "R"  # Assign label (O for Organic, R for Recyclable)
    plt.title(title, fontsize=16)

plt.show()


Convert Images to Feature Vectors  
Logistic regression requires numerical feature vectors, so we flatten images into 1D arrays.

In [ ]:
# Resize all images and flatten into feature vectors
img_size = (64, 64)  # Resize all images to 64x64
X_train = [cv2.resize(img, img_size).flatten() for img in train_img]  # Convert to 1D vectors
X_test = [cv2.resize(img, img_size).flatten() for img in test_img]  # Convert to 1D vectors

# Convert to NumPy arrays
X_train = np.array(X_train)
X_test = np.array(X_test)

print(f"Feature Vector Shape (Train): {X_train.shape}, (Test): {X_test.shape}")


Normalize the Data
Logistic regression works better when pixel values are scaled (e.g., mean=0, std=1).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


Training the Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train the model
model = LogisticRegression(max_iter=500)
model.fit(X_train, train_labels)

Evaluate the Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions
y_pred = model.predict(X_test)

# Print Accuracy and Classification Report
print(f"Accuracy: {accuracy_score(test_labels, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, y_pred))

# Confusion Matrix
cm = confusion_matrix(test_labels, y_pred)
print("\nConfusion Matrix:")
print(cm)